# 15 - D17: train with the extra field photos, then decide

**In short:** ResNet-50 is trained on `train_d17.csv` (the old training photos plus the new
field photos from notebook 14) with **exactly v1's recipe**, three times with different
seeds. Each new model is compared with v1 on the selection set (field photos from places no
model has trained on), and a rule written down **before** training decides whether the new
model replaces v1.

**Why three times:** training has luck in it; the same model trained again scores a little
differently. The spread between the three runs shows how much of a difference is luck.

**The rule (fixed 20 Oct 2026, `src/adoption.py`):**

| Stage | When | Decides |
|---|---|---|
| 1 | 3 new seeds vs v1's one seed | **Clear** if the difference is more than twice the new seeds' spread AND the paired bootstrap 95% range (same photos for both models) excludes zero: decided. Otherwise **close**. |
| 2 | only if stage 1 was close: 2 more v1 seeds, 3 vs 3 | Better if the difference is more than the larger seed spread AND the bootstrap range is above zero. |

The new model is adopted only if it is better AND validation macro-F1 is not worse (a drop
no larger than the seed spread). The measure is selection-set accuracy with the crop given,
as the app works. PlantDoc is reported only. The report states how many seeds each side had.

**Nothing of v1 changes:** v1 (`models/compare_resnet50`) is only read, and the new runs go
into new folders (`d17_resnet50_seed42/43/44`, `v1_resnet50_seed43/44`, `d17_comparison`).
v1 was also backed up on 20 Oct to `backups/v1_2026-10-20`.

**How to run:** T4 GPU. *Run all*. Each training run takes about 25 minutes; progress is saved
every epoch, so after a disconnect *Run all* again continues where it stopped, and finished
runs are skipped. If the result says **close**, set `STAGE = 2` in the settings cell and *Run
all* again.

In [1]:
from pathlib import Path
import sys

from google.colab import drive

drive.mount("/content/drive")

REPO_URL = "https://github.com/BilliartMuthoni/confidence-calibrated-crop-diagnosis.git"
REPO_DIR = Path("/content/drive/MyDrive/Sproutly-app")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

ML_DIR = REPO_DIR / "ml"
if str(ML_DIR) not in sys.path:
    sys.path.insert(0, str(ML_DIR))

import config

Mounted at /content/drive


In [2]:
# Keep the Drive copy of the repository in sync with GitHub.
!git -C {REPO_DIR} config core.fileMode false
!git -C {REPO_DIR} pull --ff-only
!git -C {REPO_DIR} status --short --branch

Already up to date.
Refresh index: 100% (318/318), done.
## main...origin/main
 M ml/notebooks/15_d17_training_and_decision.ipynb
?? ml/results/fusion_quick_test/


In [3]:
%pip install -q -r {ML_DIR}/requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 373.9/373.9 kB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 405.1/405.1 kB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 108.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 6.5 MB/s eta 0:00:00


## Settings

`RECIPE` is v1's (notebook 09, ResNet-50 at the shared settings) and is checked against v1's
saved `settings.json`. Only `STAGE` may be changed, and only when stage 1 says "close".

In [4]:
import importlib
import json
import shutil

import numpy as np
import pandas as pd
import torch

from src import adoption, calibration as cal, metrics, training
from src.data import image_dataset
for module in (training, metrics, cal, adoption, image_dataset):
    importlib.reload(module)

STAGE = 1   # 2 only when stage 1 says "close"

RECIPE = {"image_size": 224, "batch_size": 64, "epochs": 15, "patience": 4,
          "learning_rate": 1e-3, "weight_decay": 1e-2}
V1_DIR = config.MODELS_DIR / "compare_resnet50"          # v1: read only
NEW_RUNS = {f"d17_resnet50_seed{s}": s for s in (42, 43, 44)}
V1_EXTRA_RUNS = {f"v1_resnet50_seed{s}": s for s in (43, 44)}   # stage 2 only
OUT_DIR = config.MODELS_DIR / "d17_comparison"

v1_settings = json.loads((V1_DIR / "settings.json").read_text())
assert v1_settings["architecture"].split()[0] == "resnet50"
for key, value in RECIPE.items():
    assert v1_settings[key] == value, f"{key}: v1 used {v1_settings[key]}, recipe says {value}"
PROTECTED = {V1_DIR, config.MODELS_DIR / "calibration", config.MODELS_DIR / "calibration_followup",
             config.MODELS_DIR / "fusion_late_followup"}
for name in [*NEW_RUNS, *V1_EXTRA_RUNS, OUT_DIR.name]:
    assert config.MODELS_DIR / name not in PROTECTED, f"{name} would overwrite v1"

assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU, then run again."
DEVICE = "cuda"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Stage", STAGE, "| v1 recipe confirmed | GPU:", torch.cuda.get_device_name(0))

Stage 1 | v1 recipe confirmed | GPU: Tesla T4


## The photos

`train_d17.csv` must start with every row of `train.csv`, unchanged; validation, test,
PlantDoc and the selection set are the same files v1 was scored on.

In [5]:
names = ["train", "train_d17", "val", "test", "ood_test", "selection"]
splits = {name: pd.read_csv(config.SPLITS_DIR / f"{name}.csv") for name in names}
n_old = len(splits["train"])
assert splits["train_d17"]["path"].iloc[:n_old].tolist() == splits["train"]["path"].tolist(), \
    "train_d17.csv must start with train.csv unchanged"

CLASSES = list(config.PLANTVILLAGE_CLASSES)
INDEX = {name: i for i, name in enumerate(CLASSES)}
CLASS_CROPS = np.array([{"Corn_(maize)": "maize", "Tomato": "tomato", "Potato": "potato"}[c.split("___")[0]]
                        for c in CLASSES])
assert set(splits["train_d17"]["class"]) == set(CLASSES)

counts = pd.concat({"train.csv": splits["train"]["class"].value_counts(),
                    "train_d17.csv": splits["train_d17"]["class"].value_counts()}, axis=1).fillna(0).astype(int)
counts["added"] = counts["train_d17.csv"] - counts["train.csv"]
display(counts)

,train.csv,train_d17.csv,added
class,,,
Tomato___Late_blight,1617,1722,105
Tomato___Bacterial_spot,1519,1629,110
Potato___Late_blight,1404,2904,1500
Tomato___healthy,1334,1506,172
Potato___Early_blight,1284,2585,1301
Corn_(maize)___healthy,912,991,79
Corn_(maize)___Common_rust_,838,855,17
Potato___healthy,808,1449,641
Corn_(maize)___Northern_Leaf_Blight,686,686,0


In [6]:
from torch.utils.data import DataLoader

LOCAL_ROOT = Path("/content/image_cache")
all_paths = sorted(set(pd.concat(splits.values())["path"]))
cached = image_dataset.cache_images(all_paths, config.RAW_DATA_DIR, LOCAL_ROOT)
assert cached == len(all_paths), f"Only {cached} of {len(all_paths)} images cached"


def loader(rows, train):
    transform = image_dataset.train_transform(RECIPE["image_size"]) if train else image_dataset.eval_transform(RECIPE["image_size"])
    dataset = image_dataset.SplitDataset(rows, LOCAL_ROOT, CLASSES, transform)
    return DataLoader(dataset, batch_size=RECIPE["batch_size"], shuffle=train, drop_last=train,
                      num_workers=2, pin_memory=True, persistent_workers=True)


EVAL_SETS = ["val", "test", "ood_test", "selection"]
EVAL = {name: loader(splits[name], False) for name in EVAL_SETS}
print("Cached", cached, "photos")

Caching images:   0%|          | 0/25053 [00:00<?, ?it/s]

Cached 25053 photos


## Train

About 25 minutes per run on a T4. Finished runs are skipped; an interrupted run resumes.

In [7]:
def train(run_name, seed, train_rows, train_list):
    run_dir = config.MODELS_DIR / run_name
    if (run_dir / "model.pt").exists():
        print(run_name, "already trained: skipped")
        return run_dir
    run_dir.mkdir(parents=True, exist_ok=True)
    settings = {**RECIPE, "architecture": f"resnet50 (ImageNet weights: {training.ARCHITECTURES['resnet50'][1].name})",
                "seed": seed, "train_list": train_list, "gpu": torch.cuda.get_device_name(0), "torch": torch.__version__}
    training.save_json(settings, run_dir / "settings.json")
    training.seed_everything(seed)
    model = training.build_model(len(CLASSES), "resnet50")
    training.fit(model, loader(train_rows, True), EVAL["val"], run_dir, settings, DEVICE)
    model.load_state_dict(torch.load(run_dir / "best.pt", map_location=DEVICE))
    training.export(model.cpu(), CLASSES, RECIPE["image_size"], image_dataset.MEAN, image_dataset.STD,
                    run_dir / "model.pt", settings, architecture="resnet50")
    del model
    torch.cuda.empty_cache()
    return run_dir


RUNS = {"v1": V1_DIR}
for name, seed in NEW_RUNS.items():
    RUNS[name] = train(name, seed, splits["train_d17"], "train_d17.csv")
if STAGE == 2:
    for name, seed in V1_EXTRA_RUNS.items():
        RUNS[name] = train(name, seed, splits["train"], "train.csv")
print(list(RUNS))

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 184MB/s]


Training:   0%|          | 0/241 [00:00<?, ?it/s]

Predicting:   0%|          | 0/38 [00:00<?, ?it/s]

Epoch 1/15: train loss 0.7403 | val loss 0.3604 | val acc 0.8713 | val macro-F1 0.8661  (best) | 1.8 min


...

Predicting:   0%|          | 0/38 [00:00<?, ?it/s]

Epoch 14/15: train loss 0.0185 | val loss 0.0439 | val acc 0.9891 | val macro-F1 0.9867  (best) | 1.8 min


Training:   0%|          | 0/241 [00:00<?, ?it/s]

Predicting:   0%|          | 0/38 [00:00<?, ?it/s]

Epoch 15/15: train loss 0.0147 | val loss 0.0441 | val acc 0.9891 | val macro-F1 0.9874  (best) | 1.8 min
['v1', 'd17_resnet50_seed42', 'd17_resnet50_seed43', 'd17_resnet50_seed44']


## Score every model on the same photos

Raw scores are saved in `d17_comparison/`, never in v1's folder.

In [ ]:
def scores_for(run_name, run_dir):
    saved = {name: OUT_DIR / f"{run_name}__{name}.npy" for name in EVAL_SETS}
    if all(p.exists() for p in saved.values()):
        return {name: np.load(p) for name, p in saved.items()}
    model = training.build_model(len(CLASSES), "resnet50", pretrained=False)
    model.load_state_dict(torch.load(run_dir / "best.pt", map_location=DEVICE))
    model.to(DEVICE)
    out = {}
    for name in EVAL_SETS:
        out[name], _ = training.predict(model, EVAL[name], DEVICE)
        np.save(saved[name], out[name])
    del model
    torch.cuda.empty_cache()
    return out


LOGITS = {name: scores_for(name, run_dir) for name, run_dir in RUNS.items()}
LABELS = {name: splits[name]["class"].map(INDEX) for name in EVAL_SETS}


def crop_given_correct(logits, name, mask=None):
    labels = LABELS[name]
    keep = labels.notna().to_numpy() if mask is None else (labels.notna().to_numpy() & mask)
    lab = labels[keep].astype(int).to_numpy()
    _, predicted = cal.crop_confidence(metrics.softmax(logits[keep]), CLASS_CROPS, CLASS_CROPS[lab])
    return predicted == lab


FIELD = splits["val"]["source"].isin(["fieldplant", "tanzania"]).to_numpy()
rows = []
for run, out in LOGITS.items():
    val_lab = LABELS["val"].astype(int).to_numpy()
    test_lab = LABELS["test"].astype(int).to_numpy()
    rows.append({
        "run": run,
        "trained on": "train.csv" if run == "v1" or run.startswith("v1_") else "train_d17.csv",
        "val macro-F1": metrics.summarise(out["val"], val_lab, len(CLASSES))["macro_f1"],
        "val field macro-F1": metrics.summarise(out["val"][FIELD], val_lab[FIELD], len(CLASSES))["macro_f1"],
        "test macro-F1": metrics.summarise(out["test"], test_lab, len(CLASSES))["macro_f1"],
        "selection accuracy (crop given)": crop_given_correct(out["selection"], "selection").mean(),
        "PlantDoc accuracy (crop given, reported only)": crop_given_correct(out["ood_test"], "ood_test").mean(),
    })
table = pd.DataFrame(rows).set_index("run")
display(table.round(4))

Predicting:   0%|          | 0/38 [00:00<?, ?it/s]

Predicting:   0%|          | 0/39 [00:00<?, ?it/s]

Predicting:   0%|          | 0/15 [00:00<?, ?it/s]

Predicting:   0%|          | 0/60 [00:00<?, ?it/s]

Predicting:   0%|          | 0/38 [00:00<?, ?it/s]

...

Predicting:   0%|          | 0/39 [00:00<?, ?it/s]

Predicting:   0%|          | 0/15 [00:00<?, ?it/s]

Predicting:   0%|          | 0/60 [00:00<?, ?it/s]

Predicting:   0%|          | 0/38 [00:00<?, ?it/s]

Predicting:   0%|          | 0/39 [00:00<?, ?it/s]

## The decision

In [ ]:
v1_runs = ["v1"] + (list(V1_EXTRA_RUNS) if STAGE == 2 else [])
DECISION = adoption.decide(
    [crop_given_correct(LOGITS[r]["selection"], "selection") for r in NEW_RUNS],
    [crop_given_correct(LOGITS[r]["selection"], "selection") for r in v1_runs],
    [table.loc[r, "val macro-F1"] for r in NEW_RUNS],
    [table.loc[r, "val macro-F1"] for r in v1_runs],
)
print(json.dumps(DECISION, indent=1, default=float))
print()
print("OUTCOME:", DECISION["outcome"].upper(), "-", DECISION["reason"])
if DECISION["outcome"] == "close":
    print("\nNext: set STAGE = 2 in the settings cell and Run all again (finished runs are skipped).")

## Where it changed: selection set by source and class

Mean of the seeds on each side. The tomato rows are the ones D17 was meant to fix.

In [ ]:
sel = splits["selection"]
detail = []
for (source, cls), part in sel.groupby(["source", "class"]):
    mask = (sel["source"] == source).to_numpy() & (sel["class"] == cls).to_numpy()
    new = np.mean([crop_given_correct(LOGITS[r]["selection"], "selection", mask).mean() for r in NEW_RUNS])
    old = np.mean([crop_given_correct(LOGITS[r]["selection"], "selection", mask).mean() for r in v1_runs])
    detail.append({"source": source, "class": cls, "photos": int(mask.sum()), "v1": old, "new (mean of seeds)": new,
                   "change": new - old})
detail = pd.DataFrame(detail)
display(detail.round(3))

## Save the results

In [ ]:
training.save_json({**DECISION, "seeds_new": list(NEW_RUNS.values()),
                    "seeds_v1": [42] + (list(V1_EXTRA_RUNS.values()) if STAGE == 2 else []),
                    "rule": "src/adoption.py, fixed 20 Oct 2026 before training"}, OUT_DIR / f"decision_stage{STAGE}.json")
table.to_csv(OUT_DIR / f"runs_stage{STAGE}.csv")
detail.to_csv(OUT_DIR / f"selection_by_source_class_stage{STAGE}.csv", index=False)

REPO_RESULTS = ML_DIR / "results" / "d17_comparison"
REPO_RESULTS.mkdir(parents=True, exist_ok=True)
for file in [OUT_DIR / f"decision_stage{STAGE}.json", OUT_DIR / f"runs_stage{STAGE}.csv",
             OUT_DIR / f"selection_by_source_class_stage{STAGE}.csv"]:
    shutil.copy(file, REPO_RESULTS / file.name)
for run in [r for r in RUNS if r != "v1"]:
    for file in ["settings.json", "history.csv"]:
        shutil.copy(config.MODELS_DIR / run / file, REPO_RESULTS / f"{run}__{file}")
print(sorted(p.name for p in REPO_RESULTS.iterdir()))

In [ ]:
drive.flush_and_unmount()
drive.mount("/content/drive")
print("Drive synced and remounted.")

## Save to GitHub

Save the notebook first (Ctrl+S); only the results folder is committed, to a new branch.
If the outcome is **adopt**, the next notebooks rerun the late fusion (11b) and the
calibration and threshold (13, 13b) on the new model before the backend switches to it.

In [ ]:
from google.colab import userdata

from src import github_sync

shutil.copy(ML_DIR / "notebooks" / "15_d17_training_and_decision.ipynb", REPO_RESULTS / f"notebook_stage{STAGE}.ipynb")
print(github_sync.push(
    REPO_DIR,
    branch=f"colab/d17-training-stage{STAGE}",
    message=f"D17: train ResNet-50 with extra field photos and apply the adoption rule (stage {STAGE})",
    token=userdata.get("GITHUB_TOKEN"),
    name="BilliartMuthoni",
    email=userdata.get("GIT_EMAIL"),
    paths=[str(REPO_RESULTS.relative_to(REPO_DIR))],
))